In [1]:
import sys
from pathlib import Path

HERE = Path.cwd()
PROJECT_ROOT = HERE.parent
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [18]:
import pandas as pd
import sqlite3

db_path = "db.sqlite3"

In [20]:
def return_database_tables(db_path):
    with sqlite3.connect(db_path) as conn:
        tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
    return tables

In [19]:
def return_tables_values(table_name):
    with sqlite3.connect(db_path) as conn:
        values= pd.read_sql_query(f"SELECT * FROM {table_name};", conn)
    return values

In [39]:
parents = return_tables_values("tasks_parent")
tasks = return_tables_values("tasks_task")
parents_by_freq = parents[parents.freq.notnull()]
filtered_tasks = pd.merge(left=parents_by_freq, right=tasks, left_on="id", right_on="parent_id")
final_dataframe = filtered_tasks.sort_values(by="start").drop_duplicates(keep='first',subset='parent_id')
final_dataframe[['title', 'freq', 'id_y', 'start', 'description']].to_csv('out.csv', sep="$")